# TMDWF Joint CS-Kernel Effective Surface Template

This notebook is a standalone wrapper around the repository's joint downstream TMDWF CS-kernel workflow.
It reads already-generated Fourier bootstrap samples from one or more ensembles and fits a continuous `gamma_eff(x, bT)` spline surface without first running the per-ensemble CS-kernel averaging step.


## Imports / Setup

Run this notebook from the repository root, or adjust `REPO_ROOT` below.


In [ ]:
from pathlib import Path
import sys

REPO_ROOT = Path.cwd()
SRC_DIR = REPO_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from lqcd_analysis.notebook_workflows import (
    pretty_print_config,
    render_tmdwf_cs_kernel_joint_input_text,
    run_tmdwf_cs_kernel_joint_from_notebook,
    validate_tmdwf_cs_kernel_joint_notebook_config,
)


## User Inputs

These fields describe a joint CS-kernel effective-surface fit.
The workflow expects repository-native TMDWF Fourier outputs to already exist under each ensemble's `input_root`.


In [ ]:
workflow_config = {
    # Shared Fourier-output settings
    "gm": "T5",
    "eta": "eta0",
    "component": "real",
    "nstates": 2,
    "normalization_mode": "mode3",

    # Joint CS-kernel effective-surface settings
    "mu": 2.0,
    "scheme": "CG",
    "kernel_label": "LO",
    "reference_p1_gev": 1.0,
    "x_window": [0.2, 0.8],
    "x_knots": [0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8],
    "bT_knots_fm": [0.05, 0.10, 0.15, 0.20, 0.25, 0.30],
    "spline_kind": "linear",
    "plot": True,
    "progress": True,
    "progress_every": 10,

    # One entry per ensemble
    "ensembles": [
        {
            "label": "l48a060",
            "input_root": str(REPO_ROOT / "analysis_l48c64a060_m140_src5" / "4-FT-new"),
            "title_pattern": "l48c64a060_m140_fit_pz*",
            "ns": 48,
            "lattice_spacing_fm": 0.060,
            "pzrange": [1, 5],
            "bTrange": [0, 20],
        },
        {
            "label": "l64a050",
            "input_root": str(REPO_ROOT / "analysis_l64c64a050_m140_src5" / "4-FT-new"),
            "title_pattern": "l64c64a050_m140_fit_pz*",
            "ns": 64,
            "lattice_spacing_fm": 0.050,
            "pzrange": [3, 8],
            "bTrange": [0, 30],
        },
    ],

    # Output settings
    "results_dir": str(REPO_ROOT / "results_tmdwf_cs_kernel_joint"),
}
workflow_config


## Option Guide

Edit only `workflow_config` in the cell above for normal usage.

- `gm`, `eta`, `component`, `nstates`, `normalization_mode`: Select the Fourier output family to consume. These settings are shared by all ensembles in the joint fit.
- `mu`, `scheme`, `kernel_label`: Matching settings used in the type-2 correction. The first version expects one `kernel_label` per run.
- `reference_p1_gev`: Physical reference momentum scale in GeV used in the direct evolution formula. This is not a lattice momentum integer.
- `x_window`: Fit window in `x`.
- `x_knots`, `bT_knots_fm`: Optional spline knots for the continuous `gamma_eff(x,bT)` surface. If omitted, the workflow chooses knots from the observed x and physical bT values.
- `spline_kind`: Interpolation kind between knots. Use `linear` for the piecewise-linear hat basis or `cubic` for a natural cubic spline basis.
- `plot`: Whether to write one `gamma_eff(x,bT)` band plot for each `bT_knots_fm` value.
- `progress`, `progress_every`: Print bootstrap-fit progress while running. If `progress_every` is omitted, the workflow reports roughly every 5% of the bootstrap samples.
- `ensembles`: One dictionary per ensemble. Each dictionary gives the ensemble label, Fourier output root, per-pz title pattern, `Ns`, lattice spacing, and either `pzlist`/`bTlist` or inclusive `pzrange`/`bTrange`.
- `results_dir`: Output root for the joint summary, surface table, bootstrap surface samples, and diagnostics.

Expected input data shape:

- The workflow reads repository-native Fourier sample tables with one row per `(sample_id, x)` and a `q_sample` column.
- Each ensemble keeps its own physical `Pz` and physical `bT = nT * a`; the fit does not force ensembles onto a common lattice grid.
- One nuisance amplitude is eliminated for each `(ensemble, sample_id, x, bT)` group while the shared spline coefficients are fit.

What the workflow writes:

- `joint_gamma_eff/*_summary.txt`
- `joint_gamma_eff/tables/*_surface.txt`
- `joint_gamma_eff/samples/*_samples.txt`
- `joint_gamma_eff/diagnostics/*_diagnostics.txt`
- `joint_gamma_eff/plots/*_x_band.pdf`


## Validate Config

This uses the same parser as the CLI workflow, so it is a good way to confirm the text rendering and defaults before running.


In [ ]:
validated = validate_tmdwf_cs_kernel_joint_notebook_config(workflow_config)
validated


## Render Input Preview

This is the plain-text control file that the notebook helper materializes behind the scenes.


In [ ]:
input_preview = render_tmdwf_cs_kernel_joint_input_text(workflow_config)
print(input_preview)


## Run Workflow

This launches the repository-native joint CS-kernel effective-surface workflow and prints the generated artifacts.


In [ ]:
# outputs = run_tmdwf_cs_kernel_joint_from_notebook(workflow_config)
# for output in outputs:
#     print(output)


## Config Snapshot

This is useful to keep alongside saved results.


In [ ]:
print(pretty_print_config(workflow_config))
